# MoSE: Mixture of Smaller Experts (Google Colab Notebook)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/) 
[![VS Code Colab](https://img.shields.io/badge/VS%20Code%20Extension-Google.colab-blue?logo=visualstudiocode)](https://marketplace.visualstudio.com/items?itemName=Google.colab)

This notebook trains, benchmarks, and visualizes the **MoSE (Mixture of Smaller Experts)** language model.

### Two Ways to Run This Notebook:
1. **Inside VS Code / IDE:** Install the official [Google Colab Extension](https://marketplace.visualstudio.com/items?itemName=Google.colab), click **Select Kernel** at the top right, choose **Colab**, and connect to a cloud T4/A100 GPU directly from your local editor.
2. **In Browser at colab.research.google.com:** Open or upload this `.ipynb` file to Google Colab, select **Runtime > Change runtime type > T4 GPU**, and execute the cells.

## 1. Verify Cloud GPU Acceleration
Check that PyTorch sees the Colab CUDA hardware (NVIDIA T4 or A100).

In [8]:
!nvidia-smi

import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name     : {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Total VRAM      : {vram_gb:.2f} GB")

/bin/bash: line 1: nvidia-smi: command not found
PyTorch Version : 2.11.0+cpu
CUDA Available  : False


## 2. Mount Google Drive (Optional for Persistent Checkpoints)
Mounting Google Drive preserves your trained weights (`mose_python_model.pt`) even if the Colab instance shuts down.

In [3]:
try:
    from google.colab import drive
    import os
    drive.mount('/content/drive')
    drive_dir = '/content/drive/MyDrive/MoSE_Checkpoints'
    os.makedirs(drive_dir, exist_ok=True)
    print(f"Google Drive mounted. Checkpoints target: {drive_dir}")
except Exception as e:
    print(f"Skipping Google Drive mount (running locally or without colab drive module): {e}")

Mounted at /content/drive
Google Drive mounted. Checkpoints target: /content/drive/MyDrive/MoSE_Checkpoints


## 3. Workspace Setup & Dependencies
If running in Google Colab web environment, clone your repository or configure the working directory. Then install dependencies.

In [7]:
import os

# If running inside Google Colab cloud VM, navigate to /content/MoSE
if os.path.exists('/content'):
    if not os.path.exists('/content/MoSE'):
        # Clone if needed or work in /content/MoSE
        # !git clone https://github.com/YOUR_USERNAME/MoSE.git /content/MoSE
        os.makedirs('/content/MoSE', exist_ok=True)
    %cd /content/MoSE

!pip install -r requirements.txt -q

/content/MoSE
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


## 4. Data Pipeline Execution
Execute the 3-stage data preparation pipeline:
1. **Stage 1:** Stream raw Python code into Parquet files.
2. **Stage 2:** Filter lines/files and apply 50% Fill-in-the-Middle (FIM) transformation.
3. **Stage 3:** Train Byte-Pair Encoding tokenizer and generate flat `uint16` memory-mapped arrays (`train.bin`, `val.bin`).

In [ ]:
# Stage 1: Stream ~500 MB raw Python data
!python data_pipeline/download_dataset.py --target_mb 500.0

# Stage 2: Clean code and apply FIM
!python data_pipeline/preprocess_chunks.py --chunk_size_mb 50.0 --fim_rate 0.5

# Stage 3: Train BPE tokenizer and write binary memmaps
!python data_pipeline/tokenize_to_bin.py --vocab_size 10000 --train_ratio 0.9

*(Alternative: For an instant smoke test using synthetic dummy data, skip Step 4 and pass `--dummy_data` in Step 5).* 

## 5. Train MoSE Model on Colab GPU
Launch training with FP16 Automatic Mixed Precision (`torch.amp.autocast`), Cosine Warmup, and Switch Transformer auxiliary load-balancing loss.

With Colab's 16 GB T4 GPU, we can set `--batch_size 32` for increased throughput and stable routing gradients.

In [ ]:
# Launch training
!python train.py --batch_size 32 --lr 5e-4 --max_steps 15000 --checkpoint mose_python_model.pt

# Optional: Backup trained checkpoint to Google Drive
!if [ -d "/content/drive/MyDrive/MoSE_Checkpoints" ]; then cp mose_python_model.pt /content/drive/MyDrive/MoSE_Checkpoints/; cp data/binary/tokenizer.json /content/drive/MyDrive/MoSE_Checkpoints/; echo 'Backed up checkpoint to Google Drive'; fi

## 6. Evaluation & Expert Routing Analysis
Benchmark Fill-in-the-Middle (FIM) code infilling and display the ASCII expert specialization routing matrix across Python keywords.

In [ ]:
!python evaluate.py --checkpoint mose_python_model.pt

## 7. Interactive Research Dashboard (Streamlit + Plotly)
Run the interactive MoSE Explorer dashboard. Using Cloudflare Tunnel, we expose port 8501 so you can interact with the ribbon visualizer, layer heatmaps, and token generator directly from your browser.

In [ ]:
# Install cloudflared to create a secure public tunnel to Streamlit
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null

import subprocess
import time

# Start Streamlit in headless background mode
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(3)

print("Starting tunnel... Open the URL ending in .trycloudflare.com below:")
!cloudflared tunnel --url http://127.0.0.1:8501